# TFG V5 — Domain-wall interference

This version proposes a different approach from V4. Instead of marking a window directly by its absolute cost `C(i)`, it looks at transitions between neighboring windows. The hypothesis is that, when occupied cells appear in blocks, the boundaries between bad and good regions contain useful information.

## Quantities used

For each window `i`:

```text
C(i) = number of ones in window_i
valid(i) = 1 if C(i)=0, and 0 otherwise
Delta C(i) = C(i+1) - C(i)
Delta valid(i) = valid(i+1) - valid(i)
```

A large change or a change in validity indicates a boundary, or `domain wall`, between regions with different behavior.

## Circuit flow

1. Prepare a uniform superposition over window indices.
2. Compute the cost and transition profile classically and modularly.
3. Encode a diagonal phase on `idx` according to the selected mode:
   - `delta_cost`
   - `descent`
   - `valid_boundary`
   - `oriented_valid`
4. Apply a local mixer over neighboring indices.
5. Repeat the block and analyze whether `P_valid`, `P_boundary`, or `P_near_valid` increases.

## Purpose of this version

V5 checks whether exploiting domain walls can be a useful preprocessing phase for finding valid or near-valid windows. It does not use Grover or global diffusion; it keeps the comparison with V4 by using the same type of local mixer.

In [1]:
import os
os.environ.setdefault("MPLCONFIGDIR", os.path.join(os.getcwd(), ".matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", os.path.join(os.getcwd(), ".cache"))

from math import prod
import numpy as np
import matplotlib.pyplot as plt

import qiskit
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.quantum_info import Statevector
from qiskit.circuit.library import PhaseGate, UnitaryGate

print(qiskit.__version__)


2.3.1


In [2]:
# =========================================================
# Configurable parameters for the main experiment
# =========================================================

# Main 1D case with occupied blocks and a central free region.
D = 1
N = [8]
M = [2]
occupied_coords = [(0,), (1,), (2,), (6,), (7,)]

# Parameters de interferencia domain-wall.
theta = np.pi / 3
mixer_angle = 0.60
repetitions = 4
transition_mode = "oriented_valid"  # "delta_cost", "valid_boundary", "descent", "oriented_valid"

use_transition_phase = True
use_mixer = True
use_statevector_analysis = True
show_plots = True
mixer_method = "linear_valid"  # "local_geometric", "linear_valid", "rx_all"


## Implemented approach

In this version, the transition phase is implemented as a diagonal phase compiled directly on the `idx` register. Since the `grid` is fixed, the values `C(i)`, `valid(i)`, `Delta C(i)`, and `Delta valid(i)` are computed classically before building the circuit.

This keeps the notebook lightweight and easy to experiment with: it allows testing the `domain wall interference` hypothesis without adding reversible arithmetic for cost differences inside the circuit yet.

The mixer remains local. It can mix consecutive indices (`linear_valid`) or neighboring starting positions in the ND geometry (`local_geometric`). This avoids Grover-style global diffusion and keeps the approach comparable to V4.

In [3]:
# =========================================================
# ND geometry and classical utilities reutilizadas de V4
# =========================================================

def validate_problem(N, M):
    if len(N) != len(M):
        raise ValueError("N and M must have the same dimension.")
    for d, (n_d, m_d) in enumerate(zip(N, M)):
        if n_d <= 0 or m_d <= 0:
            raise ValueError(f"N[{d}] and M[{d}] must be positive.")
        if m_d > n_d:
            raise ValueError(f"M[{d}] cannot be greater than N[{d}].")


def coord_to_index(coord, dims):
    """Converts ND coordinates to a row-major linear index."""
    idx_lin = 0
    stride = 1
    for d in reversed(range(len(dims))):
        idx_lin += coord[d] * stride
        stride *= dims[d]
    return idx_lin


def index_to_coord(index, dims):
    """Converts a row-major linear index to ND coordinates."""
    coord = [0] * len(dims)
    rem = index
    for d in reversed(range(len(dims))):
        coord[d] = rem % dims[d]
        rem //= dims[d]
    return tuple(coord)


def valid_starts_nd(N, M):
    """Valid starting coordinates for a window M inside N."""
    return list(np.ndindex(tuple(N[d] - M[d] + 1 for d in range(len(N)))))


def window_qubits_nd(start, N, M):
    """Linear indices of the cells covered by the window starting at start."""
    qubits = []
    for offset in np.ndindex(tuple(M)):
        coord = tuple(start[d] + offset[d] for d in range(len(N)))
        qubits.append(coord_to_index(coord, N))
    return qubits


def normalize_coord(coord, D):
    if D == 1 and isinstance(coord, int):
        return (coord,)
    return tuple(coord)


def build_grid_bits(N, occupied_coords):
    """Returns a classical vector with 1 on occupied cells and 0 on free cells."""
    D = len(N)
    grid = [0] * prod(N)
    for raw_coord in occupied_coords:
        coord = normalize_coord(raw_coord, D)
        if len(coord) != D:
            raise ValueError(f"Coordinate {coord} does not have dimension {D}.")
        for d, x in enumerate(coord):
            if x < 0 or x >= N[d]:
                raise ValueError(f"Coordinate {coord} is outside the grid N={N}.")
        grid[coord_to_index(coord, N)] = 1
    return grid


def compute_window_cost_classical(grid_bits, start, N, M):
    """C(i): number of ones in the window associated with start."""
    return sum(grid_bits[q] for q in window_qubits_nd(start, N, M))


def window_string_classical(grid_bits, start, N, M):
    return ''.join(str(grid_bits[q]) for q in window_qubits_nd(start, N, M))


def valid_indicator(cost):
    return 1 if cost == 0 else 0


def get_valid_indices(grid_bits, starts, N, M):
    return [i for i, start in enumerate(starts) if compute_window_cost_classical(grid_bits, start, N, M) == 0]


def gray_order_valid(W, IDX):
    gray_full = [t ^ (t >> 1) for t in range(2**IDX)]
    return [g for g in gray_full if g < W]


def format_nd_array_from_bits(bitstring, dims):
    arr = np.array(list(bitstring), dtype=str).reshape(tuple(dims))
    return np.array2string(arr, separator=' ').replace("'", "")


In [4]:
# =========================================================
# Domain walls: costs, transitions, and phase profiles
# =========================================================

def compute_costs_and_valids(grid_bits, starts, N, M):
    costs = [compute_window_cost_classical(grid_bits, start, N, M) for start in starts]
    valids = [valid_indicator(c) for c in costs]
    return costs, valids


def transition_features(costs, valids):
    """Computes Delta C(i), Delta valid(i), and boundary flags by index."""
    W = len(costs)
    delta_cost = [None] * W
    delta_valid = [None] * W
    edge_boundary = [False] * W
    boundary_index = [False] * W

    for i in range(W - 1):
        delta_cost[i] = costs[i + 1] - costs[i]
        delta_valid[i] = valids[i + 1] - valids[i]
        edge_boundary[i] = valids[i + 1] != valids[i]
        if edge_boundary[i]:
            boundary_index[i] = True
            boundary_index[i + 1] = True

    near_valid = [c <= 1 for c in costs]
    return {
        "delta_cost": delta_cost,
        "delta_valid": delta_valid,
        "edge_boundary": edge_boundary,
        "boundary_index": boundary_index,
        "near_valid": near_valid,
    }


def transition_phase_profile(costs, valids, theta, transition_mode):
    """
    Returns a phase phi_i for each index i.

    Modos:
    - delta_cost: phi_i = theta * (C(i+1)-C(i)).
    - descent: phi_i = +theta if C decreases toward i+1, -theta if it increases.
    - valid_boundary: phi_i = theta if i is on a validity boundary.
    - oriented_valid: phi_i = +theta for invalid indices neighboring valid ones,
      and -theta for valid indices on the boundary.
    """
    features = transition_features(costs, valids)
    W = len(costs)
    phases = [0.0] * W

    if transition_mode == "delta_cost":
        for i, dc in enumerate(features["delta_cost"]):
            phases[i] = 0.0 if dc is None else theta * dc

    elif transition_mode == "descent":
        for i, dc in enumerate(features["delta_cost"]):
            if dc is None or dc == 0:
                phases[i] = 0.0
            elif dc < 0:
                phases[i] = theta
            else:
                phases[i] = -theta

    elif transition_mode == "valid_boundary":
        for i, is_boundary in enumerate(features["boundary_index"]):
            phases[i] = theta if is_boundary else 0.0

    elif transition_mode == "oriented_valid":
        for i in range(W):
            has_valid_neighbor = False
            has_invalid_neighbor = False
            for j in (i - 1, i + 1):
                if 0 <= j < W:
                    has_valid_neighbor |= valids[j] == 1
                    has_invalid_neighbor |= valids[j] == 0
            if valids[i] == 0 and has_valid_neighbor:
                phases[i] = theta
            elif valids[i] == 1 and has_invalid_neighbor:
                phases[i] = -theta
            else:
                phases[i] = 0.0

    else:
        raise ValueError(
            "transition_mode debe ser 'delta_cost', 'valid_boundary', 'descent' u 'oriented_valid'."
        )

    return phases, features


def boundary_indices_from_features(features):
    return [i for i, is_boundary in enumerate(features["boundary_index"]) if is_boundary]


def near_valid_indices_from_features(features):
    return [i for i, is_near in enumerate(features["near_valid"]) if is_near]


In [5]:
# =========================================================
# Quantum blocks: transition phase + local mixer
# =========================================================

def prepare_valid_index_superposition(qc, idx, W):
    """Prepares 1/sqrt(W) sum_{i=0}^{W-1} |i>, even when W is not a power of 2."""
    IDX = len(idx)
    amps = np.zeros(2**IDX, dtype=complex)
    amps[:W] = 1 / np.sqrt(W)
    qc.initialize(amps, idx)


def apply_phase_to_basis_state(qc, idx, basis_state, angle):
    """Applies exp(i angle) to the computational state |basis_state> of the idx register."""
    if abs(angle) < 1e-15:
        return

    IDX = len(idx)
    zero_qubits = [q for bit, q in enumerate(idx) if ((basis_state >> bit) & 1) == 0]
    for q in zero_qubits:
        qc.x(q)

    if IDX == 1:
        qc.p(angle, idx[0])
    else:
        gate = PhaseGate(angle).control(IDX - 1)
        qc.append(gate, list(idx[:-1]) + [idx[-1]])

    for q in reversed(zero_qubits):
        qc.x(q)


def apply_transition_phase(qc, idx, phases, W, label="U_DW"):
    """Applies a diagonal phase compiled from the domain-wall profile."""
    for i in range(W):
        apply_phase_to_basis_state(qc, idx, i, phases[i])
    qc.barrier(label=label)


def two_level_mixer_gate(num_qubits, a, b, beta, label=None):
    """Local rotation exp(-i beta X_ab) in span{|a>, |b>}."""
    dim = 2**num_qubits
    U = np.eye(dim, dtype=complex)
    c = np.cos(beta)
    s = -1j * np.sin(beta)
    U[a, a] = c
    U[b, b] = c
    U[a, b] = s
    U[b, a] = s
    return UnitaryGate(U, label=label or f"Mix({a},{b})")


def mixer_edges_from_starts(starts, N, method="local_geometric"):
    """Local edges between valid indices. Does not implement Grover global diffusion."""
    if method == "linear_valid":
        return [(i, i + 1) for i in range(len(starts) - 1)]

    if method != "local_geometric":
        raise ValueError("Unknown local mixer method.")

    start_to_idx = {tuple(s): i for i, s in enumerate(starts)}
    edges = []
    D = len(N)
    for i, start in enumerate(starts):
        for d in range(D):
            neigh = list(start)
            neigh[d] += 1
            neigh = tuple(neigh)
            if neigh in start_to_idx:
                edges.append((i, start_to_idx[neigh]))
    return edges


def apply_local_mixer(qc, idx, starts, N, mixer_angle, method="local_geometric"):
    """
    Modular local mixer on idx.
    - local_geometric: mixes neighboring starting positions in the ND grid.
    - linear_valid: mixes consecutive indices.
    - rx_all: simple prototype; it can leak amplitude to idx states >= W when W is not a power of two.
    """
    if abs(mixer_angle) < 1e-15:
        return

    if method == "rx_all":
        for q in idx:
            qc.rx(2 * mixer_angle, q)
        return

    IDX = len(idx)
    for a, b in mixer_edges_from_starts(starts, N, method):
        qc.append(two_level_mixer_gate(IDX, a, b, mixer_angle), list(idx))


In [6]:
# =========================================================
# Circuit construction, analysis, and visualization
# =========================================================

def build_domain_wall_circuit(
    N, M, occupied_coords, theta, mixer_angle, repetitions,
    transition_mode="oriented_valid",
    use_transition_phase=True,
    use_mixer=True,
    mixer_method="local_geometric",
    add_barriers=True,
):
    validate_problem(N, M)
    starts = valid_starts_nd(N, M)
    W = len(starts)
    IDX = int(np.ceil(np.log2(W))) if W > 1 else 1
    grid_bits = build_grid_bits(N, occupied_coords)
    costs, valids = compute_costs_and_valids(grid_bits, starts, N, M)
    phases, features = transition_phase_profile(costs, valids, theta, transition_mode)

    idx = QuantumRegister(IDX, "i")
    qc = QuantumCircuit(idx)
    prepare_valid_index_superposition(qc, idx, W)
    if add_barriers:
        qc.barrier(label="init")

    for r in range(repetitions):
        if use_transition_phase:
            apply_transition_phase(qc, idx, phases, W, label=f"U_DW[{r}]")
        if use_mixer:
            apply_local_mixer(qc, idx, starts, N, mixer_angle, method=mixer_method)
        if add_barriers:
            qc.barrier(label=f"layer {r}")

    metadata = {
        "N": list(N),
        "M": list(M),
        "D": len(N),
        "starts": starts,
        "W": W,
        "IDX": IDX,
        "grid_bits": grid_bits,
        "occupied_coords": [normalize_coord(c, len(N)) for c in occupied_coords],
        "costs": costs,
        "valids": valids,
        "phases": phases,
        "features": features,
        "theta": theta,
        "mixer_angle": mixer_angle,
        "repetitions": repetitions,
        "transition_mode": transition_mode,
        "mixer_method": mixer_method,
    }
    return qc, metadata


def index_probabilities_from_statevector(sv, metadata):
    IDX = metadata["IDX"]
    probs = np.zeros(2**IDX, dtype=float)
    idx_mask = (1 << IDX) - 1
    for basis_idx, amp in enumerate(sv.data):
        probs[basis_idx & idx_mask] += float(abs(amp) ** 2)
    return probs


def analyze_probabilities(sv, metadata, title="Domain-wall analysis", tol=1e-12):
    N, M = metadata["N"], metadata["M"]
    starts = metadata["starts"]
    W = metadata["W"]
    costs = metadata["costs"]
    valids = metadata["valids"]
    features = metadata["features"]
    grid_bits = metadata["grid_bits"]
    probs = index_probabilities_from_statevector(sv, metadata)

    valid_indices = [i for i, v in enumerate(valids) if v == 1]
    boundary_indices = boundary_indices_from_features(features)
    near_valid_indices = near_valid_indices_from_features(features)

    p_valid_initial = len(valid_indices) / W
    p_boundary_initial = len(boundary_indices) / W
    p_near_valid_initial = len(near_valid_indices) / W

    p_valid_after = float(sum(probs[i] for i in valid_indices))
    p_boundary_after = float(sum(probs[i] for i in boundary_indices))
    p_near_valid_after = float(sum(probs[i] for i in near_valid_indices))
    p_invalid_index_after = float(sum(probs[i] for i in range(W, len(probs))))

    print()
    print(f"============ {title} ============")
    print(f"N={N}, M={M}, W={W}, IDX={metadata['IDX']}")
    print(f"theta={metadata['theta']:.6g}, mixer_angle={metadata['mixer_angle']:.6g}, repetitions={metadata['repetitions']}")
    print(f"transition_mode={metadata['transition_mode']}, mixer_method={metadata['mixer_method']}")
    print(f"valid_indices={valid_indices}")
    print(f"boundary_indices={boundary_indices}")
    print(f"near_valid_indices={near_valid_indices}")
    print(f"P_valid_initial      = {p_valid_initial:.6f}")
    print(f"P_valid_after        = {p_valid_after:.6f}")
    print(f"P_boundary_initial   = {p_boundary_initial:.6f}")
    print(f"P_boundary_after     = {p_boundary_after:.6f}")
    print(f"P_near_valid_initial = {p_near_valid_initial:.6f}")
    print(f"P_near_valid_after   = {p_near_valid_after:.6f}")
    if p_invalid_index_after > tol:
        print(f"P_invalid_index_after = {p_invalid_index_after:.6f}")

    print()
    print("index | start coordinate | window | C(i) | valid(i) | Delta C | Delta valid | phase | probability")
    print("------|------------------|--------|------|----------|---------|-------------|-------|------------")
    for i, start in enumerate(starts):
        window = window_string_classical(grid_bits, start, N, M)
        dc = features["delta_cost"][i]
        dv = features["delta_valid"][i]
        dc_str = "-" if dc is None else str(dc)
        dv_str = "-" if dv is None else str(dv)
        print(
            f"{i:>5} | {str(start):<16} | {window:<6} | {costs[i]:>4} | {valids[i]:>8} | "
            f"{dc_str:>7} | {dv_str:>11} | {metadata['phases'][i]:>5.2f} | {probs[i]:>10.6f}"
        )

    return {
        "probs": probs,
        "P_valid_initial": p_valid_initial,
        "P_valid_after": p_valid_after,
        "P_boundary_initial": p_boundary_initial,
        "P_boundary_after": p_boundary_after,
        "P_near_valid_initial": p_near_valid_initial,
        "P_near_valid_after": p_near_valid_after,
        "P_invalid_index_after": p_invalid_index_after,
        "valid_indices": valid_indices,
        "boundary_indices": boundary_indices,
        "near_valid_indices": near_valid_indices,
    }


def plot_domain_wall_results(metadata, analysis, title="Domain-wall results"):
    W = metadata["W"]
    x = np.arange(W)
    initial_probs = np.ones(W) / W
    after_probs = analysis["probs"][:W]
    costs = np.array(metadata["costs"])
    valids = set(analysis["valid_indices"])
    boundaries = set(analysis["boundary_indices"])

    fig, axes = plt.subplots(2, 2, figsize=(12, 7))
    fig.suptitle(title)

    axes[0, 0].bar(x - 0.18, initial_probs, width=0.36, label="inicial")
    axes[0, 0].bar(x + 0.18, after_probs, width=0.36, label="after")
    axes[0, 0].set_title("Probability by index")
    axes[0, 0].set_xlabel("index i")
    axes[0, 0].set_ylabel("probability")
    axes[0, 0].legend()

    axes[0, 1].plot(x, costs, marker="o")
    axes[0, 1].set_title("Coste C(i)")
    axes[0, 1].set_xlabel("index i")
    axes[0, 1].set_ylabel("C(i)")

    colors = ["tab:green" if i in valids else "tab:red" for i in x]
    axes[1, 0].bar(x, after_probs, color=colors, alpha=0.75)
    for i in boundaries:
        axes[1, 0].axvline(i, color="black", linestyle="--", alpha=0.45)
    axes[1, 0].set_title("Validos y fronteras")
    axes[1, 0].set_xlabel("index i")
    axes[1, 0].set_ylabel("final probability")

    labels = ["valid", "boundary", "near_valid"]
    initial = [analysis["P_valid_initial"], analysis["P_boundary_initial"], analysis["P_near_valid_initial"]]
    final = [analysis["P_valid_after"], analysis["P_boundary_after"], analysis["P_near_valid_after"]]
    pos = np.arange(len(labels))
    axes[1, 1].bar(pos - 0.18, initial, width=0.36, label="inicial")
    axes[1, 1].bar(pos + 0.18, final, width=0.36, label="after")
    axes[1, 1].set_xticks(pos)
    axes[1, 1].set_xticklabels(labels)
    axes[1, 1].set_ylim(0, 1.05)
    axes[1, 1].set_title("Metricas agregadas")
    axes[1, 1].legend()

    plt.tight_layout()
    plt.show()


def run_domain_wall_experiment(
    name, N, M, occupied_coords, theta, mixer_angle, repetitions,
    transition_mode="oriented_valid",
    mixer_method="local_geometric",
    use_transition_phase=True,
    use_mixer=True,
    use_statevector_analysis=True,
    show_plot=False,
):
    print()
    print("#" * 40)
    print(f"Experiment: {name}")
    print("#" * 40)

    qc, metadata = build_domain_wall_circuit(
        N=N,
        M=M,
        occupied_coords=occupied_coords,
        theta=theta,
        mixer_angle=mixer_angle,
        repetitions=repetitions,
        transition_mode=transition_mode,
        use_transition_phase=use_transition_phase,
        use_mixer=use_mixer,
        mixer_method=mixer_method,
    )

    analysis = None
    if use_statevector_analysis:
        sv = Statevector.from_instruction(qc)
        analysis = analyze_probabilities(sv, metadata, title=name)
        if show_plot:
            plot_domain_wall_results(metadata, analysis, title=name)

    return qc, metadata, analysis


## Small tests

These cases check that the notebook runs in 1D and 2D, and that the domain-wall approach can measure whether `P_valid`, `P_boundary`, or `P_near_valid` increase. A universal improvement is not expected: the goal is to provide a modular benchmark for sweeping `theta`, `mixer_angle`, `repetitions`, and `transition_mode`.

In [7]:
qc_test_block, meta_test_block, analysis_test_block = run_domain_wall_experiment(
    name="Test 1D domain wall: blocks",
    N=[8],
    M=[3],
    occupied_coords=[(0,), (1,), (2,), (6,), (7,)],
    theta=np.pi / 2,
    mixer_angle=0.30,
    repetitions=3,
    transition_mode="valid_boundary",
    mixer_method="linear_valid",
    show_plot=False,
)

qc_test_2d, meta_test_2d, analysis_test_2d = run_domain_wall_experiment(
    name="2D domain-wall test: 4x4 with 2x2 window",
    N=[4, 4],
    M=[2, 2],
    occupied_coords=[(0, 0), (0, 1), (1, 0), (3, 3), (3, 2), (2, 3)],
    theta=np.pi / 6,
    mixer_angle=0.25,
    repetitions=2,
    transition_mode="descent",
    mixer_method="local_geometric",
    show_plot=False,
)



########################################
Experiment: Test 1D domain wall: blocks
########################################

============ Test 1D domain wall: blocks ============
N=[8], M=[3], W=6, IDX=3
theta=1.5708, mixer_angle=0.3, repetitions=3
transition_mode=valid_boundary, mixer_method=linear_valid
valid_indices=[3]
boundary_indices=[2, 3, 4]
near_valid_indices=[2, 3, 4]
P_valid_initial      = 0.166667
P_valid_after        = 0.256956
P_boundary_initial   = 0.500000
P_boundary_after     = 0.402469
P_near_valid_initial = 0.500000
P_near_valid_after   = 0.402469

index | start coordinate | window | C(i) | valid(i) | Delta C | Delta valid | phase | probability
------|------------------|--------|------|----------|---------|-------------|-------|------------
    0 | (0,)             | 111    |    3 |        0 |      -1 |           0 |  0.00 |   0.224243
    1 | (1,)             | 110    |    2 |        0 |      -1 |           0 |  0.00 |   0.106459
    2 | (2,)             | 100    |  

## Design notes

- `TFG_V5` does not use Grover or a global diffusion operator. Interference appears by combining domain-wall phases with a local mixer.
- The transition phase is encoded as a diagonal phase on `idx`, using costs computed from the fixed grid. This is a compact oracle-like version used to test the hypothesis before implementing full reversible arithmetic for `Delta C`.
- `P_boundary_after` measures whether the circuit concentrates amplitude on boundaries between valid and invalid regions. `P_near_valid_after` measures whether it concentrates amplitude on windows with `C(i)<=1`, which can be useful near-solution candidates.
- The modes `delta_cost`, `descent`, `valid_boundary`, and `oriented_valid` allow comparing different interpretations of a domain wall.

## Unified case-study analysis

The main experiment above is intentionally kept as a manual sandbox: change `N`, `M`, `occupied_coords`, `theta`, `mixer_angle`, `repetitions`, and the mixer there when you want to run quick tests.

This section defines a shared ordered suite of 10 case studies, from very small 1D instances to larger 1D, 2D, and small 3D instances. For every case it applies the same analysis pipeline: repetition oscillation, two-parameter heatmap, spectrum/eigenvalue period study, theta scan, and a final dashboard with the optimal probabilities. The numerical sweeps use the exact `W`-dimensional index-space layer model, which is equivalent to the phase-plus-local-mixer dynamics and avoids repeatedly simulating auxiliary registers.


In [8]:
# =========================================================
# Unified case-study analysis for TFG V5
# =========================================================
# This block supersedes the previous single-instance Analysis 1-6 cells.
# The main experiment remains above for quick manual tests; the study cases
# below are the reproducible benchmark set for thesis figures.

import csv
import re
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except Exception:
    pass

plt.rcParams.update({
    "axes.titlesize": 14,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
})

VALID_COLOR = "#2ecc71"
INVALID_COLOR = "#e74c3c"
BASELINE_COLOR = "0.45"
V5_OUTPUT_DIR = Path("analysis_v5_cases")
V5_OUTPUT_DIR.mkdir(exist_ok=True)

# Full analysis settings. Increase these for finer publication sweeps.
V5_CASE_MAX_REPS = 60
V5_GRID_POINTS = 31
V5_THETA_SCAN_POINTS = 121
V5_SPECTRAL_REPS = 80
V5_TRANSITION_MODES = ["delta_cost", "descent", "valid_boundary", "oriented_valid"]

V5_CASE_STUDIES = [
    {
        "name": "01_1d_tiny_single_gap",
        "description": "Minimal 1D case with one valid window.",
        "N": [6], "M": [2],
        "occupied_coords": [(0,), (3,), (4,)],
        "theta": np.pi / 2, "beta": 0.30, "mixer_method": "local_geometric",
    },
    {
        "name": "02_1d_main_reference",
        "description": "Reference 1D instance used in the manual main experiment.",
        "N": [8], "M": [2],
        "occupied_coords": [(0,), (1,), (2,), (6,), (7,)],
        "theta": np.pi / 3, "beta": 0.60, "mixer_method": "linear_valid",
    },
    {
        "name": "03_1d_two_free_regions",
        "description": "1D grid with two occupied blocks and a central valid plateau.",
        "N": [10], "M": [3],
        "occupied_coords": [(0,), (1,), (7,), (8,), (9,)],
        "theta": np.pi / 3, "beta": 0.30, "mixer_method": "linear_valid",
    },
    {
        "name": "04_1d_clustered_medium",
        "description": "Medium 1D case with several clustered obstacles.",
        "N": [16], "M": [3],
        "occupied_coords": [(0,), (1,), (5,), (6,), (7,), (13,), (14,)],
        "theta": np.pi / 3, "beta": 0.24, "mixer_method": "local_geometric",
    },
    {
        "name": "05_1d_long_clustered_blocks",
        "description": "Longer 1D case with separated occupied blocks.",
        "N": [32], "M": [4],
        "occupied_coords": [(0,), (1,), (2,), (9,), (10,), (11,), (18,), (19,), (28,), (29,), (30,), (31,)],
        "theta": np.pi / 4, "beta": 0.18, "mixer_method": "local_geometric",
    },
    {
        "name": "06_2d_tiny_corner_block",
        "description": "Small 2D case with a single valid 2x2 region.",
        "N": [3, 3], "M": [2, 2],
        "occupied_coords": [(0, 0), (0, 2), (2, 2)],
        "theta": np.pi / 2, "beta": 0.28, "mixer_method": "local_geometric",
    },
    {
        "name": "07_2d_small_diagonal_block",
        "description": "4x4 grid with diagonal obstacles and one clear lower-left solution.",
        "N": [4, 4], "M": [2, 2],
        "occupied_coords": [(1, 1), (2, 2), (0, 3)],
        "theta": np.pi / 2, "beta": 0.24, "mixer_method": "local_geometric",
    },
    {
        "name": "08_2d_medium_clustered_obstacles",
        "description": "5x5 grid with two compact occupied clusters.",
        "N": [5, 5], "M": [2, 2],
        "occupied_coords": [(0, 0), (0, 1), (1, 0), (3, 3), (3, 4), (4, 3)],
        "theta": np.pi / 3, "beta": 0.22, "mixer_method": "local_geometric",
    },
    {
        "name": "09_2d_rectangular_window",
        "description": "6x6 grid with a non-square 3x2 window.",
        "N": [6, 6], "M": [3, 2],
        "occupied_coords": [(0, 0), (0, 1), (1, 0), (4, 4), (4, 5), (5, 4), (2, 3)],
        "theta": np.pi / 3, "beta": 0.18, "mixer_method": "local_geometric",
    },
    {
        "name": "10_3d_small_clustered_obstacles",
        "description": "Small 3D grid with two compact occupied clusters.",
        "N": [4, 4, 3], "M": [2, 2, 2],
        "occupied_coords": [(0, 0, 0), (0, 0, 1), (1, 0, 0), (3, 3, 2), (3, 2, 2), (2, 3, 2)],
        "theta": np.pi / 4, "beta": 0.16, "mixer_method": "local_geometric",
    },
]


def v5_slug(text):
    return re.sub(r"[^a-zA-Z0-9_]+", "_", text).strip("_").lower()


def v5_savefig(fig, stem):
    pdf = V5_OUTPUT_DIR / f"{stem}.pdf"
    png = V5_OUTPUT_DIR / f"{stem}.png"
    fig.savefig(pdf, bbox_inches="tight")
    fig.savefig(png, dpi=200, bbox_inches="tight")
    plt.close(fig)


def v5_case_context(case):
    N_case = list(case["N"])
    M_case = list(case["M"])
    occupied_case = list(case["occupied_coords"])
    validate_problem(N_case, M_case)
    starts_case = valid_starts_nd(N_case, M_case)
    grid_bits_case = build_grid_bits(N_case, occupied_case)
    costs_case, valids_case = compute_costs_and_valids(grid_bits_case, starts_case, N_case, M_case)
    valid_indices_case = [i for i, v in enumerate(valids_case) if v == 1]
    if not valid_indices_case:
        raise ValueError(f"Case {case['name']} has no valid windows; change occupied_coords.")
    features_case = transition_features(costs_case, valids_case)
    W_case = len(starts_case)
    return {
        "name": case["name"],
        "description": case.get("description", ""),
        "N": N_case,
        "M": M_case,
        "occupied_coords": occupied_case,
        "starts": starts_case,
        "grid_bits": grid_bits_case,
        "costs": costs_case,
        "valids": valids_case,
        "features": features_case,
        "valid_indices": valid_indices_case,
        "boundary_indices": boundary_indices_from_features(features_case),
        "near_valid_indices": near_valid_indices_from_features(features_case),
        "W": W_case,
        "P_uniform": len(valid_indices_case) / W_case,
        "theta_default": float(case.get("theta", globals().get("theta", np.pi / 3))),
        "beta_default": float(case.get("beta", globals().get("mixer_angle", 0.25))),
        "mixer_method": case.get("mixer_method", "local_geometric"),
    }


def v5_two_level_matrix(W, a, b, beta):
    U = np.eye(W, dtype=complex)
    c = np.cos(beta)
    s = -1j * np.sin(beta)
    U[a, a] = c
    U[b, b] = c
    U[a, b] = s
    U[b, a] = s
    return U


def v5_phase_vector(ctx, theta_value, transition_mode):
    phases_case, _features_case = transition_phase_profile(ctx["costs"], ctx["valids"], theta_value, transition_mode)
    return np.asarray(phases_case, dtype=float)


def v5_layer_matrix(ctx, theta_value, beta_value, transition_mode):
    W_case = ctx["W"]
    D = np.diag(np.exp(1j * v5_phase_vector(ctx, theta_value, transition_mode)))
    U_mix = np.eye(W_case, dtype=complex)
    edges = mixer_edges_from_starts(ctx["starts"], ctx["N"], ctx["mixer_method"])
    for a, b in edges:
        U_mix = v5_two_level_matrix(W_case, a, b, beta_value) @ U_mix
    return U_mix @ D


def v5_initial_state(ctx):
    return np.ones(ctx["W"], dtype=complex) / np.sqrt(ctx["W"])


def v5_probabilities(ctx, theta_value, beta_value, reps_value, transition_mode):
    U = v5_layer_matrix(ctx, theta_value, beta_value, transition_mode)
    psi = v5_initial_state(ctx)
    for _ in range(int(reps_value)):
        psi = U @ psi
    return np.abs(psi) ** 2


def v5_p_valid(ctx, probs):
    return float(np.sum(np.asarray(probs)[ctx["valid_indices"]]))


def v5_p_boundary(ctx, probs):
    return float(np.sum(np.asarray(probs)[ctx["boundary_indices"]])) if ctx["boundary_indices"] else 0.0


def v5_p_near_valid(ctx, probs):
    return float(np.sum(np.asarray(probs)[ctx["near_valid_indices"]])) if ctx["near_valid_indices"] else 0.0


def v5_oscillation_curve(ctx, theta_value, beta_value, transition_mode, max_reps=V5_CASE_MAX_REPS):
    U = v5_layer_matrix(ctx, theta_value, beta_value, transition_mode)
    psi = v5_initial_state(ctx)
    reps = np.arange(1, max_reps + 1)
    values = []
    for _r in reps:
        psi = U @ psi
        values.append(v5_p_valid(ctx, np.abs(psi) ** 2))
    return reps, np.asarray(values, dtype=float)


def v5_heatmap(ctx, r_star, transition_mode):
    theta_values = np.linspace(0.0, np.pi, V5_GRID_POINTS)
    beta_values = np.linspace(0.0, np.pi / 2, V5_GRID_POINTS)
    grid = np.zeros((len(theta_values), len(beta_values)))
    for i, theta_value in enumerate(theta_values):
        for j, beta_value in enumerate(beta_values):
            grid[i, j] = v5_p_valid(ctx, v5_probabilities(ctx, theta_value, beta_value, r_star, transition_mode))
    return theta_values, beta_values, grid


def v5_theta_scan(ctx, beta_value, r_star, transition_mode):
    theta_values = np.linspace(0.0, 2 * np.pi, V5_THETA_SCAN_POINTS)
    values = np.asarray([v5_p_valid(ctx, v5_probabilities(ctx, t, beta_value, r_star, transition_mode)) for t in theta_values])
    return theta_values, values


def v5_spectral_curve(ctx, theta_value, beta_value, transition_mode, max_reps=V5_SPECTRAL_REPS):
    U = v5_layer_matrix(ctx, theta_value, beta_value, transition_mode)
    eigvals, eigvecs = np.linalg.eig(U)
    coeff = np.linalg.solve(eigvecs, v5_initial_state(ctx))
    reps = np.arange(1, max_reps + 1)
    direct = []
    spectral = []
    psi = v5_initial_state(ctx)
    for r in reps:
        psi = U @ psi
        direct.append(v5_p_valid(ctx, np.abs(psi) ** 2))
        psi_spec = eigvecs @ ((eigvals ** r) * coeff)
        spectral.append(v5_p_valid(ctx, np.abs(psi_spec) ** 2))
    return eigvals, reps, np.asarray(direct), np.asarray(spectral)


def v5_plot_case_mode(ctx, transition_mode):
    prefix = f"v5_{v5_slug(ctx['name'])}_{transition_mode}"
    theta_default = ctx["theta_default"]
    beta_default = ctx["beta_default"]

    reps, p_curve = v5_oscillation_curve(ctx, theta_default, beta_default, transition_mode)
    r_star = int(reps[int(np.argmax(p_curve))])
    p_star_default = float(np.max(p_curve))

    theta_grid, beta_grid, heat = v5_heatmap(ctx, r_star, transition_mode)
    best_i, best_j = np.unravel_index(int(np.argmax(heat)), heat.shape)
    theta_opt = float(theta_grid[best_i])
    beta_opt = float(beta_grid[best_j])
    p_opt = float(heat[best_i, best_j])

    theta_scan_values, theta_scan_probs = v5_theta_scan(ctx, beta_default, r_star, transition_mode)
    theta_scan_idx = int(np.argmax(theta_scan_probs))
    theta_scan_opt = float(theta_scan_values[theta_scan_idx])
    p_theta_scan_opt = float(theta_scan_probs[theta_scan_idx])

    eigvals, spectral_reps, direct_curve, spectral_curve = v5_spectral_curve(ctx, theta_default, beta_default, transition_mode)
    opt_probs = v5_probabilities(ctx, theta_opt, beta_opt, r_star, transition_mode)
    p_boundary_opt = v5_p_boundary(ctx, opt_probs)
    p_near_valid_opt = v5_p_near_valid(ctx, opt_probs)

    # Analysis 1: oscillation curve.
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(reps, p_curve, marker="o", linewidth=2, color=VALID_COLOR, label="P_valid")
    ax.axhline(ctx["P_uniform"], color=BASELINE_COLOR, linestyle="--", linewidth=1.5, label="uniform K/W")
    ax.axvline(r_star, color="black", linestyle=":", linewidth=1.5, label=f"r*={r_star}")
    ax.annotate(f"max={p_star_default:.3f}", xy=(r_star, p_star_default), xytext=(r_star + 1, min(1.0, p_star_default + 0.08)), arrowprops={"arrowstyle": "->", "lw": 1.0})
    ax.set_xlabel("repetitions r")
    ax.set_ylabel("P_valid")
    ax.set_title(f"V5 oscillation - {ctx['name']} - {transition_mode}")
    ax.legend()
    fig.tight_layout()
    v5_savefig(fig, f"{prefix}_01_oscillation")

    # Analysis 2: heatmap.
    fig, ax = plt.subplots(figsize=(6, 4))
    im = ax.imshow(heat, origin="lower", aspect="auto", cmap="viridis", extent=[beta_grid[0] / np.pi, beta_grid[-1] / np.pi, theta_grid[0] / np.pi, theta_grid[-1] / np.pi])
    ax.scatter([beta_opt / np.pi], [theta_opt / np.pi], marker="*", s=180, color="white", edgecolor="black", linewidth=0.7, label="optimum")
    ax.set_xlabel("beta / pi")
    ax.set_ylabel("theta / pi")
    ax.set_title(f"V5 P_valid heatmap at r={r_star} - {ctx['name']} - {transition_mode}")
    ax.legend(loc="lower right")
    fig.colorbar(im, ax=ax, label="P_valid")
    fig.tight_layout()
    v5_savefig(fig, f"{prefix}_02_heatmap")

    # Analysis 3: eigenvalue spectrum and period prediction.
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    unit = np.exp(1j * np.linspace(0, 2 * np.pi, 400))
    axes[0].plot(unit.real, unit.imag, color="0.75", linewidth=1.2)
    sc = axes[0].scatter(eigvals.real, eigvals.imag, c=eigvals.imag, cmap="viridis", s=45)
    axes[0].set_aspect("equal", adjustable="box")
    axes[0].set_xlabel("Re(lambda)")
    axes[0].set_ylabel("Im(lambda)")
    axes[0].set_title("One-layer spectrum")
    fig.colorbar(sc, ax=axes[0], label="Im(lambda)")
    axes[1].plot(spectral_reps, direct_curve, color=VALID_COLOR, linewidth=2, label="direct layer chain")
    axes[1].plot(spectral_reps, spectral_curve, color="black", linestyle="--", linewidth=1.4, label="eigen reconstruction")
    axes[1].axhline(ctx["P_uniform"], color=BASELINE_COLOR, linestyle="--", linewidth=1.5, label="uniform K/W")
    axes[1].set_xlabel("repetitions r")
    axes[1].set_ylabel("P_valid")
    axes[1].set_title("Period prediction")
    axes[1].legend()
    fig.suptitle(f"V5 eigenvalue analysis - {ctx['name']} - {transition_mode}", fontsize=15)
    fig.tight_layout()
    v5_savefig(fig, f"{prefix}_03_spectrum")

    # Analysis 4: theta scan.
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(theta_scan_values / np.pi, theta_scan_probs, color=VALID_COLOR, linewidth=2, label="P_valid")
    ax.axhline(ctx["P_uniform"], color=BASELINE_COLOR, linestyle="--", linewidth=1.5, label="uniform K/W")
    ax.axvline(theta_scan_opt / np.pi, color="black", linestyle=":", linewidth=1.5, label=f"theta*={theta_scan_opt / np.pi:.3f} pi")
    ax.set_xlabel("theta / pi")
    ax.set_ylabel("P_valid")
    ax.set_title(f"V5 theta scan at beta={beta_default:.3f}, r={r_star} - {transition_mode}")
    ax.legend()
    fig.tight_layout()
    v5_savefig(fig, f"{prefix}_04_theta_scan")

    # Analysis 5: summary dashboard.
    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    axes[0, 0].plot(reps, p_curve, marker="o", linewidth=2, color=VALID_COLOR, label="P_valid")
    axes[0, 0].axhline(ctx["P_uniform"], color=BASELINE_COLOR, linestyle="--", linewidth=1.5, label="uniform K/W")
    axes[0, 0].axvline(r_star, color="black", linestyle=":", linewidth=1.2)
    axes[0, 0].set_xlabel("repetitions r")
    axes[0, 0].set_ylabel("P_valid")
    axes[0, 0].set_title("Oscillation")
    axes[0, 0].legend()
    im = axes[0, 1].imshow(heat, origin="lower", aspect="auto", cmap="viridis", extent=[beta_grid[0] / np.pi, beta_grid[-1] / np.pi, theta_grid[0] / np.pi, theta_grid[-1] / np.pi])
    axes[0, 1].scatter([beta_opt / np.pi], [theta_opt / np.pi], marker="*", s=180, color="white", edgecolor="black", linewidth=0.7)
    axes[0, 1].set_xlabel("beta / pi")
    axes[0, 1].set_ylabel("theta / pi")
    axes[0, 1].set_title("Heatmap")
    fig.colorbar(im, ax=axes[0, 1], label="P_valid")
    axes[1, 0].plot(theta_scan_values / np.pi, theta_scan_probs, color=VALID_COLOR, linewidth=2)
    axes[1, 0].axhline(ctx["P_uniform"], color=BASELINE_COLOR, linestyle="--", linewidth=1.5)
    axes[1, 0].axvline(theta_scan_opt / np.pi, color="black", linestyle=":", linewidth=1.2)
    axes[1, 0].set_xlabel("theta / pi")
    axes[1, 0].set_ylabel("P_valid")
    axes[1, 0].set_title("Theta scan")
    bar_colors = [VALID_COLOR if i in ctx["valid_indices"] else INVALID_COLOR for i in range(ctx["W"])]
    axes[1, 1].bar(np.arange(ctx["W"]), opt_probs, color=bar_colors)
    axes[1, 1].axhline(1 / ctx["W"], color=BASELINE_COLOR, linestyle="--", linewidth=1.5, label="uniform 1/W")
    axes[1, 1].set_xlabel("window index")
    axes[1, 1].set_ylabel("probability")
    axes[1, 1].set_title("Optimal distribution")
    axes[1, 1].legend()
    fig.suptitle(f"V5 case summary - {ctx['name']} - {transition_mode}", fontsize=16)
    fig.tight_layout(rect=[0, 0.02, 1, 0.96])
    v5_savefig(fig, f"{prefix}_05_summary")

    return {
        "version": "V5",
        "case": ctx["name"],
        "transition_mode": transition_mode,
        "description": ctx["description"],
        "N": str(ctx["N"]),
        "M": str(ctx["M"]),
        "W": ctx["W"],
        "valid_windows": len(ctx["valid_indices"]),
        "boundary_windows": len(ctx["boundary_indices"]),
        "near_valid_windows": len(ctx["near_valid_indices"]),
        "P_uniform": ctx["P_uniform"],
        "theta_default": theta_default,
        "beta_default": beta_default,
        "r_star_default": r_star,
        "P_star_default": p_star_default,
        "theta_opt_heatmap": theta_opt,
        "beta_opt_heatmap": beta_opt,
        "P_opt_heatmap": p_opt,
        "P_boundary_at_optimum": p_boundary_opt,
        "P_near_valid_at_optimum": p_near_valid_opt,
        "theta_opt_scan_default_beta": theta_scan_opt,
        "P_opt_scan_default_beta": p_theta_scan_opt,
        "mixer_method": ctx["mixer_method"],
    }


def v5_plot_transition_mode_comparison_for_case(case_name, rows):
    """Grafico comparativo de los cuatro transition modes para un use case."""
    case_rows = [row for row in rows if row["case"] == case_name]
    if not case_rows:
        return

    row_by_mode = {row["transition_mode"]: row for row in case_rows}
    modes = [mode for mode in V5_TRANSITION_MODES if mode in row_by_mode]
    if not modes:
        return

    p_uniform = row_by_mode[modes[0]]["P_uniform"]
    p_default = [row_by_mode[mode]["P_star_default"] for mode in modes]
    p_opt = [row_by_mode[mode]["P_opt_heatmap"] for mode in modes]

    x = np.arange(len(modes))
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x - 0.18, p_default, width=0.36, color="#3498db", alpha=0.85, label="default theta/beta")
    ax.bar(x + 0.18, p_opt, width=0.36, color=VALID_COLOR, alpha=0.85, label="best scanned P_valid")
    ax.axhline(p_uniform, color=BASELINE_COLOR, linestyle="--", linewidth=1.5, label="uniform K/W")
    for xi, value in zip(x + 0.18, p_opt):
        ax.text(xi, min(1.04, value + 0.025), f"{value:.3f}", ha="center", va="bottom", fontsize=10)
    ax.set_xticks(x)
    ax.set_xticklabels(modes, rotation=20, ha="right")
    ax.set_ylabel("P_valid")
    ax.set_ylim(0, 1.10)
    ax.set_title(f"V5 transition mode comparison - {case_name}")
    ax.legend()
    fig.tight_layout()
    v5_savefig(fig, f"v5_{v5_slug(case_name)}_06_transition_mode_comparison")


def v5_plot_transition_mode_mean_comparison(rows):
    """Grafico global con medias por transition mode en todos los use cases."""
    modes = [mode for mode in V5_TRANSITION_MODES if any(row["transition_mode"] == mode for row in rows)]
    if not modes:
        return

    mean_default = []
    mean_opt = []
    mean_gain = []
    for mode in modes:
        mode_rows = [row for row in rows if row["transition_mode"] == mode]
        mean_default.append(float(np.mean([row["P_star_default"] for row in mode_rows])))
        mean_opt.append(float(np.mean([row["P_opt_heatmap"] for row in mode_rows])))
        mean_gain.append(float(np.mean([row["P_opt_heatmap"] - row["P_uniform"] for row in mode_rows])))

    x = np.arange(len(modes))
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x - 0.24, mean_default, width=0.24, color="#3498db", alpha=0.85, label="mean default")
    ax.bar(x, mean_opt, width=0.24, color=VALID_COLOR, alpha=0.85, label="mean best scanned")
    ax.bar(x + 0.24, mean_gain, width=0.24, color="#f39c12", alpha=0.85, label="mean gain vs uniform")
    for xi, value in zip(x, mean_opt):
        ax.text(xi, min(1.04, value + 0.025), f"{value:.3f}", ha="center", va="bottom", fontsize=10)
    ax.axhline(0.0, color=BASELINE_COLOR, linestyle="--", linewidth=1.0)
    ax.set_xticks(x)
    ax.set_xticklabels(modes, rotation=20, ha="right")
    ax.set_ylabel("mean P_valid")
    ax.set_ylim(min(0.0, min(mean_gain) - 0.05), 1.10)
    ax.set_title("V5 default-mixer transition mode mean comparison")
    ax.legend()
    fig.tight_layout()
    v5_savefig(fig, "v5_default_mixer_transition_mode_mean_comparison")


def v5_run_all_case_studies():
    rows = []
    start = time.time()
    for case in V5_CASE_STUDIES:
        try:
            ctx = v5_case_context(case)
            print(f"[V5] Case {ctx['name']}: N={ctx['N']}, M={ctx['M']}, W={ctx['W']}, valid={len(ctx['valid_indices'])}, uniform={ctx['P_uniform']:.4f}")
            for transition_mode in case.get("transition_modes", V5_TRANSITION_MODES):
                row = v5_plot_case_mode(ctx, transition_mode)
                rows.append(row)
                print(f"      {transition_mode}: P_valid={row['P_opt_heatmap']:.6f}, theta={row['theta_opt_heatmap']:.6f}, beta={row['beta_opt_heatmap']:.6f}, r={row['r_star_default']}")
            v5_plot_transition_mode_comparison_for_case(ctx["name"], rows)
        except Exception as exc:
            print(f"[V5] Case skipped {case.get('name', '<unknown>')}: {exc}")

    if not rows:
        print("[V5] No case studies completed.")
        return []

    csv_name = V5_OUTPUT_DIR / "v5_case_study_summary.csv"
    with open(csv_name, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

    best_rows = []
    for case in V5_CASE_STUDIES:
        case_rows = [row for row in rows if row["case"] == case["name"]]
        if case_rows:
            best_rows.append(max(case_rows, key=lambda row: row["P_opt_heatmap"]))

    labels = [row["case"].replace("_", "\n", 2) for row in best_rows]
    x = np.arange(len(best_rows))
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(x - 0.18, [row["P_uniform"] for row in best_rows], width=0.36, color="0.7", label="uniform K/W")
    ax.bar(x + 0.18, [row["P_opt_heatmap"] for row in best_rows], width=0.36, color=VALID_COLOR, label="best scanned P_valid")
    for xi, row in zip(x, best_rows):
        ax.text(xi + 0.18, min(1.02, row["P_opt_heatmap"] + 0.025), row["transition_mode"], rotation=90, ha="center", va="bottom", fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_ylabel("P_valid")
    ax.set_ylim(0, 1.12)
    ax.set_title("V5 default-mixer case-study heatmap overview")
    ax.legend()
    fig.tight_layout()
    v5_savefig(fig, "v5_default_mixer_case_study_overview")

    v5_plot_transition_mode_mean_comparison(rows)

    print(f"[V5] Completed {len(rows)} case/mode analyses in {time.time() - start:.1f}s")
    print(f"[V5] Summary CSV and figures saved in: {V5_OUTPUT_DIR}")
    return rows


v5_case_study_results = v5_run_all_case_studies()


# =========================================================
# ANALYSIS 6 — Parameter optimization methods across all case studies
# =========================================================
import csv
import time
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg", force=True)
import matplotlib.pyplot as plt

FAST_MODE = False  # set True to reduce sweep for large N

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except Exception:
    pass

plt.rcParams.update({
    "axes.titlesize": 14,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
})

PARAM_OUTPUT_DIR = Path("analysis_v5_cases")
PARAM_OUTPUT_DIR.mkdir(exist_ok=True)

M1_THETA_POINTS = 10 if FAST_MODE else 25
M1_BETA_POINTS = 10 if FAST_MODE else 25
M1_R_MAX = 15 if FAST_MODE else 29
M2_R_MAX = 60
M4_MAX_ITERS = 100 if FAST_MODE else 200
HYBRID_REFINE_POINTS = 5 if FAST_MODE else 9
METHOD_MODES = ["delta_cost", "descent", "valid_boundary", "oriented_valid"]
MIXER_METHODS_V5 = ["local_geometric", "linear_valid", "rx_all"]


def v5_opt_context(case, mixer_method=None):
    N_case = list(case["N"])
    M_case = list(case["M"])
    occupied_case = list(case["occupied_coords"])
    validate_problem(N_case, M_case)
    starts_case = valid_starts_nd(N_case, M_case)
    grid_bits_case = build_grid_bits(N_case, occupied_case)
    costs_case, valids_case = compute_costs_and_valids(grid_bits_case, starts_case, N_case, M_case)
    valid_indices_case = [i for i, v in enumerate(valids_case) if v == 1]
    if not valid_indices_case:
        raise ValueError(f"Case {case['name']} has no valid windows.")
    W_case = len(starts_case)
    IDX_case = int(np.ceil(np.log2(W_case))) if W_case > 1 else 1
    DIM_case = 2 ** IDX_case
    mixer_case = mixer_method or case.get("mixer_method", "local_geometric")
    return {
        "name": case["name"],
        "description": case.get("description", ""),
        "N": N_case,
        "M": M_case,
        "starts": starts_case,
        "grid_bits": grid_bits_case,
        "costs": costs_case,
        "valids": valids_case,
        "valid_indices": valid_indices_case,
        "W": W_case,
        "IDX": IDX_case,
        "DIM": DIM_case,
        "theta_default": float(case.get("theta", globals().get("theta", np.pi / 3))),
        "beta_default": float(case.get("beta", globals().get("mixer_angle", 0.25))),
        "mixer_method": mixer_case,
        "P_uniform": len(valid_indices_case) / W_case,
    }


def v5_opt_gate_matrix(ctx, a, b, beta_value):
    return np.asarray(two_level_mixer_gate(ctx["IDX"], a, b, beta_value).to_matrix(), dtype=complex)


def v5_opt_phase(ctx, theta_value, mode):
    phases, _features = transition_phase_profile(ctx["costs"], ctx["valids"], theta_value, mode)
    padded = np.zeros(ctx["DIM"], dtype=float)
    padded[:ctx["W"]] = np.asarray(phases, dtype=float)
    return padded


def v5_opt_rx_all_matrix(ctx, beta_value):
    c = np.cos(beta_value)
    s = -1j * np.sin(beta_value)
    rx = np.asarray([[c, s], [s, c]], dtype=complex)
    U = np.asarray([[1.0 + 0.0j]], dtype=complex)
    for _ in range(ctx["IDX"]):
        U = np.kron(U, rx)
    return U


def v5_mixer_edges_for_analysis(ctx):
    if ctx["mixer_method"] == "rx_all":
        return [(a, b) for a in range(ctx["W"]) for b in range(a + 1, ctx["W"]) if bin(a ^ b).count("1") == 1]
    return mixer_edges_from_starts(ctx["starts"], ctx["N"], ctx["mixer_method"])


def v5_opt_mixer(ctx, beta_value):
    if ctx["mixer_method"] == "rx_all":
        return v5_opt_rx_all_matrix(ctx, beta_value)
    U_mix = np.eye(ctx["DIM"], dtype=complex)
    for a, b in v5_mixer_edges_for_analysis(ctx):
        U_mix = v5_opt_gate_matrix(ctx, a, b, beta_value) @ U_mix
    return U_mix


def v5_opt_layer(ctx, theta_value, beta_value, mode):
    D = np.diag(np.exp(1j * v5_opt_phase(ctx, theta_value, mode)))
    return v5_opt_mixer(ctx, beta_value) @ D


def v5_opt_initial(ctx):
    psi = np.zeros(ctx["DIM"], dtype=complex)
    psi[:ctx["W"]] = 1.0 / np.sqrt(ctx["W"])
    return psi


def v5_opt_p_from_state(ctx, psi):
    return float(np.sum(np.abs(psi[ctx["valid_indices"]]) ** 2))


def v5_opt_p_valid(ctx, theta_value, beta_value, r_value, mode):
    U = v5_opt_layer(ctx, theta_value, beta_value, mode)
    psi = v5_opt_initial(ctx)
    for _ in range(int(r_value)):
        psi = U @ psi
    return v5_opt_p_from_state(ctx, psi)


def v5_opt_curve(ctx, theta_value, beta_value, mode, r_max=M2_R_MAX):
    U = v5_opt_layer(ctx, theta_value, beta_value, mode)
    psi = v5_opt_initial(ctx)
    reps = np.arange(1, r_max + 1)
    values = []
    for _ in reps:
        psi = U @ psi
        values.append(v5_opt_p_from_state(ctx, psi))
    return reps, np.asarray(values, dtype=float)


def v5_existing_experimental(ctx, mode):
    for row in globals().get("v5_case_study_results", []):
        if row.get("case") == ctx["name"] and row.get("transition_mode") == mode and row.get("mixer_method") == ctx["mixer_method"]:
            return {
                "method": "Experimental",
                "theta": float(row["theta_opt_heatmap"]),
                "beta": float(row["beta_opt_heatmap"]),
                "r": int(row["r_star_default"]),
                "P": float(row["P_opt_heatmap"]),
            }
    return None


def v5_method1(ctx, mode):
    theta_values = np.linspace(0.0, np.pi, M1_THETA_POINTS)
    beta_values = np.linspace(0.0, np.pi / 2, M1_BETA_POINTS)
    best = {"method": "M1 (sweep)", "theta": np.nan, "beta": np.nan, "r": np.nan, "P": -np.inf}
    t0 = time.perf_counter()
    for ti, theta_value in enumerate(theta_values):
        if ti % 5 == 0:
            print(f"[V5 M1] {ctx['name']} / {mode}: theta point {ti + 1}/{len(theta_values)}")
        D = np.diag(np.exp(1j * v5_opt_phase(ctx, theta_value, mode)))
        for beta_value in beta_values:
            U = v5_opt_mixer(ctx, beta_value) @ D
            psi = v5_opt_initial(ctx)
            for r_value in range(1, M1_R_MAX + 1):
                psi = U @ psi
                p_value = v5_opt_p_from_state(ctx, psi)
                if p_value > best["P"]:
                    best = {"method": "M1 (sweep)", "theta": float(theta_value), "beta": float(beta_value), "r": int(r_value), "P": float(p_value)}
    print(f"[V5 M1] {ctx['name']} / {mode} completed in {time.perf_counter() - t0:.2f}s")
    return best


def v5_method2(ctx, mode):
    U = v5_opt_layer(ctx, ctx["theta_default"], ctx["beta_default"], mode)
    eigenvalues, V = np.linalg.eig(U)
    try:
        coeffs = np.linalg.solve(V, v5_opt_initial(ctx))
    except np.linalg.LinAlgError:
        coeffs = np.linalg.pinv(V) @ v5_opt_initial(ctx)
    reps = np.arange(1, M2_R_MAX + 1)
    analytical = []
    direct = []
    psi_direct = v5_opt_initial(ctx)
    for r_value in reps:
        psi_direct = U @ psi_direct
        direct.append(v5_opt_p_from_state(ctx, psi_direct))
        psi_eig = V @ ((eigenvalues ** r_value) * coeffs)
        analytical.append(v5_opt_p_from_state(ctx, psi_eig))
    analytical = np.asarray(analytical, dtype=float)
    direct = np.asarray(direct, dtype=float)
    deviation = float(np.max(np.abs(analytical - direct)))
    idx = int(np.argmax(analytical))
    return {
        "method": "M2 (eigen)",
        "theta": ctx["theta_default"],
        "beta": ctx["beta_default"],
        "r": int(reps[idx]),
        "P": float(analytical[idx]),
        "eig_deviation": deviation,
    }


def v5_method3(ctx, mode):
    valid_set = set(ctx["valid_indices"])
    invalid_counts = []
    for a in ctx["valid_indices"]:
        count = 0
        for u, v in v5_mixer_edges_for_analysis(ctx):
            other = v if u == a else u if v == a else None
            if other is not None and other not in valid_set:
                count += 1
        invalid_counts.append(count)
    n_avg = float(np.mean(invalid_counts)) if invalid_counts else 1.0
    theta_theory = np.pi / 2
    beta_theory = np.arctan(n_avg)
    return {"method": "M3 (theory)", "theta": float(theta_theory), "beta": float(beta_theory), "r": 1, "P": v5_opt_p_valid(ctx, theta_theory, beta_theory, 1, mode)}


def v5_method4(ctx, mode, r_fixed):
    theta_value = np.pi / 4
    beta_value = np.pi / 6
    history = []
    for _ in range(M4_MAX_ITERS):
        p_now = v5_opt_p_valid(ctx, theta_value, beta_value, r_fixed, mode)
        history.append(p_now)
        g_theta = (v5_opt_p_valid(ctx, theta_value + np.pi / 2, beta_value, r_fixed, mode) - v5_opt_p_valid(ctx, theta_value - np.pi / 2, beta_value, r_fixed, mode)) / 2
        g_beta = (v5_opt_p_valid(ctx, theta_value, beta_value + np.pi / 2, r_fixed, mode) - v5_opt_p_valid(ctx, theta_value, beta_value - np.pi / 2, r_fixed, mode)) / 2
        if np.sqrt(g_theta ** 2 + g_beta ** 2) < 1e-4:
            break
        theta_value = (theta_value + 0.05 * g_theta) % (2 * np.pi)
        beta_value = abs(beta_value + 0.05 * g_beta) % np.pi
    p_final = v5_opt_p_valid(ctx, theta_value, beta_value, r_fixed, mode)
    history.append(p_final)
    return {"method": "M4 (grad)", "theta": float(theta_value), "beta": float(beta_value), "r": int(r_fixed), "P": float(p_final), "grad_iters": len(history)}


def v5_best_r_for_layer(ctx, U, r_max=M2_R_MAX):
    psi = v5_opt_initial(ctx)
    best_r = 1
    best_p = -np.inf
    for r_value in range(1, int(r_max) + 1):
        psi = U @ psi
        p_value = v5_opt_p_from_state(ctx, psi)
        if p_value > best_p:
            best_r = int(r_value)
            best_p = float(p_value)
    return best_r, best_p


def v5_method5(ctx, mode):
    theta_values = np.linspace(0.0, np.pi, M1_THETA_POINTS)
    beta_values = np.linspace(0.0, np.pi / 2, M1_BETA_POINTS)
    best = {"method": "M5 (hybrid)", "theta": np.nan, "beta": np.nan, "r": np.nan, "P": -np.inf}
    t0 = time.perf_counter()

    def evaluate(theta_value, beta_value):
        U = v5_opt_layer(ctx, theta_value, beta_value, mode)
        r_value, p_value = v5_best_r_for_layer(ctx, U, M2_R_MAX)
        return {"method": "M5 (hybrid)", "theta": float(theta_value), "beta": float(beta_value), "r": int(r_value), "P": float(p_value)}

    seeds = [(ctx["theta_default"], ctx["beta_default"])]
    if mode == "oriented_valid":
        seeds.append((np.pi / 2, np.pi / 4))
    try:
        theory = v5_method3(ctx, mode)
        seeds.append((theory["theta"], theory["beta"]))
    except Exception:
        pass

    for ti, theta_value in enumerate(theta_values):
        if ti % 5 == 0:
            print(f"[V5 M5] {ctx['name']} / {mode}: theta point {ti + 1}/{len(theta_values)}")
        for beta_value in beta_values:
            candidate = evaluate(theta_value, beta_value)
            if candidate["P"] > best["P"]:
                best = candidate
    for theta_value, beta_value in seeds:
        candidate = evaluate(theta_value, beta_value)
        if candidate["P"] > best["P"]:
            best = candidate

    theta_step = np.pi / max(1, M1_THETA_POINTS - 1)
    beta_step = (np.pi / 2) / max(1, M1_BETA_POINTS - 1)
    theta_refine = np.linspace(max(0.0, best["theta"] - theta_step), min(np.pi, best["theta"] + theta_step), HYBRID_REFINE_POINTS)
    beta_refine = np.linspace(max(0.0, best["beta"] - beta_step), min(np.pi / 2, best["beta"] + beta_step), HYBRID_REFINE_POINTS)
    for theta_value in theta_refine:
        for beta_value in beta_refine:
            candidate = evaluate(theta_value, beta_value)
            if candidate["P"] > best["P"]:
                best = candidate
    best["hybrid_seconds"] = time.perf_counter() - t0
    print(f"[V5 M5] {ctx['name']} / {mode} completed in {best['hybrid_seconds']:.2f}s")
    return best


def v5_fallback_experimental(ctx, mode):
    reps, curve = v5_opt_curve(ctx, ctx["theta_default"], ctx["beta_default"], mode, M2_R_MAX)
    r_star = int(reps[int(np.argmax(curve))])
    theta_values = np.linspace(0.0, np.pi, M1_THETA_POINTS)
    beta_values = np.linspace(0.0, np.pi / 2, M1_BETA_POINTS)
    best = {"method": "Experimental", "theta": ctx["theta_default"], "beta": ctx["beta_default"], "r": r_star, "P": float(np.max(curve))}
    for theta_value in theta_values:
        for beta_value in beta_values:
            p_value = v5_opt_p_valid(ctx, theta_value, beta_value, r_star, mode)
            if p_value > best["P"]:
                best = {"method": "Experimental", "theta": float(theta_value), "beta": float(beta_value), "r": r_star, "P": float(p_value)}
    return best


def v5_record(ctx, mode, result, experimental):
    return {
        "case": ctx["name"],
        "transition_mode": mode,
        "mixer_method": ctx["mixer_method"],
        "method": result["method"],
        "theta": result["theta"],
        "theta_over_pi": result["theta"] / np.pi,
        "beta": result["beta"],
        "beta_over_pi": result["beta"] / np.pi,
        "r": result["r"],
        "P_valid": result["P"],
        "P_uniform": ctx["P_uniform"],
        "delta_theta_pi_vs_exp": (result["theta"] - experimental["theta"]) / np.pi,
        "delta_beta_pi_vs_exp": (result["beta"] - experimental["beta"]) / np.pi,
        "delta_r_vs_exp": int(result["r"] - experimental["r"]),
        "delta_P_vs_exp": result["P"] - experimental["P"],
        "W": ctx["W"],
        "valid_windows": len(ctx["valid_indices"]),
    }

all_rows_v5 = []
for case in V5_CASE_STUDIES:
    try:
        modes = case.get("transition_modes", METHOD_MODES)
        for mixer_method in MIXER_METHODS_V5:
            ctx = v5_opt_context(case, mixer_method=mixer_method)
            print(f"\n[V5 Analysis 6] Case {ctx['name']} | mixer={ctx['mixer_method']} | W={ctx['W']} | valid={len(ctx['valid_indices'])} | uniform={ctx['P_uniform']:.4f}")
            for mode in modes:
                print(f"[V5 Analysis 6] Mode {mode}")
                m1 = v5_method1(ctx, mode)
                m2 = v5_method2(ctx, mode)
                print(f"Eigenvalue method verified for {ctx['name']} / {ctx['mixer_method']} / {mode}: max deviation = {m2.get('eig_deviation', np.nan):.3e}")
                m3 = v5_method3(ctx, mode)
                m4 = v5_method4(ctx, mode, int(m2["r"]))
                m5 = v5_method5(ctx, mode)
                experimental = v5_existing_experimental(ctx, mode) or v5_fallback_experimental(ctx, mode)
                for result in [m1, m2, m3, m4, m5, experimental]:
                    all_rows_v5.append(v5_record(ctx, mode, result, experimental))
    except Exception as exc:
        print(f"WARNING: V5 Analysis 6 skipped case {case.get('name', '<unknown>')}: {exc}")

csv_path = PARAM_OUTPUT_DIR / "v5_param_methods_all_cases.csv"
if all_rows_v5:
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(all_rows_v5[0].keys()))
        writer.writeheader()
        writer.writerows(all_rows_v5)

    methods = ["M1 (sweep)", "M2 (eigen)", "M3 (theory)", "M4 (grad)", "M5 (hybrid)", "Experimental"]
    cases = [case["name"] for case in V5_CASE_STUDIES]
    # For readability, the figure shows the best transition mode per case and method.
    summary_rows = []
    for case_name in cases:
        for method in methods:
            candidates = [row for row in all_rows_v5 if row["case"] == case_name and row["method"] == method]
            if candidates:
                summary_rows.append(max(candidates, key=lambda row: row["P_valid"]))

    x = np.arange(len(cases))
    width = 0.13
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for mi, method in enumerate(methods):
        method_rows = [next(row for row in summary_rows if row["case"] == case and row["method"] == method) for case in cases]
        offset = (mi - (len(methods) - 1) / 2) * width
        axes[0].bar(x + offset, [row["P_valid"] for row in method_rows], width=width, label=method)
        axes[1].bar(x + offset, [row["theta_over_pi"] for row in method_rows], width=width, label=method)
        axes[2].bar(x + offset, [row["beta_over_pi"] for row in method_rows], width=width, label=method)
    for ax, title, ylabel in zip(axes, ["Best P_valid by method", "theta/pi by method", "beta/pi by method"], ["P_valid", "theta/pi", "beta/pi"]):
        ax.set_xticks(x)
        ax.set_xticklabels([c.replace("_", "\n", 2) for c in cases], rotation=45, ha="right")
        ax.set_title(title)
        ax.set_ylabel(ylabel)
    axes[0].legend(fontsize=8)
    fig.suptitle("V5 parameter optimization methods across all case studies\n(best transition mode and mixer per case/method; full table in CSV)", fontsize=15)
    fig.tight_layout(rect=[0, 0.03, 1, 0.92])
    fig.savefig(PARAM_OUTPUT_DIR / "v5_param_methods_all_cases.pdf", bbox_inches="tight")
    fig.savefig(PARAM_OUTPUT_DIR / "v5_param_methods_all_cases.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

    best_rows_v5 = []
    for case_name in cases:
        candidates = [row for row in all_rows_v5 if row["case"] == case_name]
        if candidates:
            best_rows_v5.append(max(candidates, key=lambda row: row["P_valid"]))

    best_csv = PARAM_OUTPUT_DIR / "v5_all_config_best_by_case.csv"
    with open(best_csv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(best_rows_v5[0].keys()))
        writer.writeheader()
        writer.writerows(best_rows_v5)

    combo_rows = []
    for mode in METHOD_MODES:
        for mixer_method in MIXER_METHODS_V5:
            for method in methods:
                combo = [row for row in all_rows_v5 if row["transition_mode"] == mode and row["mixer_method"] == mixer_method and row["method"] == method]
                if not combo:
                    continue
                values = np.asarray([row["P_valid"] for row in combo], dtype=float)
                combo_rows.append({
                    "transition_mode": mode,
                    "mixer_method": mixer_method,
                    "method": method,
                    "mean_P_valid": float(np.mean(values)),
                    "median_P_valid": float(np.median(values)),
                    "std_P_valid": float(np.std(values)),
                    "min_P_valid": float(np.min(values)),
                    "max_P_valid": float(np.max(values)),
                    "wins_best_by_case": int(sum(row["transition_mode"] == mode and row["mixer_method"] == mixer_method and row["method"] == method for row in best_rows_v5)),
                    "cases": len(combo),
                })
    combo_csv = PARAM_OUTPUT_DIR / "v5_all_config_summary_by_mode_mixer_method.csv"
    with open(combo_csv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(combo_rows[0].keys()))
        writer.writeheader()
        writer.writerows(combo_rows)

    fig, ax = plt.subplots(figsize=(13, 5))
    ax.bar(x - 0.18, [row["P_uniform"] for row in best_rows_v5], width=0.36, color="0.7", label="uniform K/W")
    ax.bar(x + 0.18, [row["P_valid"] for row in best_rows_v5], width=0.36, color=VALID_COLOR, label="best full sweep")
    for xi, row in zip(x, best_rows_v5):
        ax.text(xi + 0.18, min(1.04, row["P_valid"] + 0.025), f"{row['transition_mode']}\n{row['mixer_method']}\n{row['method']}", rotation=90, ha="center", va="bottom", fontsize=7)
    ax.set_xticks(x)
    ax.set_xticklabels([c.replace("_", "\n", 2) for c in cases], rotation=45, ha="right")
    ax.set_ylabel("P_valid")
    ax.set_ylim(0, 1.12)
    ax.set_title("V5 full sweep best configuration by use case")
    ax.legend()
    fig.tight_layout()
    fig.savefig(PARAM_OUTPUT_DIR / "v5_case_study_overview.pdf", bbox_inches="tight")
    fig.savefig(PARAM_OUTPUT_DIR / "v5_case_study_overview.png", dpi=200, bbox_inches="tight")
    fig.savefig(PARAM_OUTPUT_DIR / "v5_all_config_best_by_case.pdf", bbox_inches="tight")
    fig.savefig(PARAM_OUTPUT_DIR / "v5_all_config_best_by_case.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

    def v5_mean_best_for_pair(row_key, col_key, row_values, col_values):
        heat = np.full((len(row_values), len(col_values)), np.nan)
        for i, row_value in enumerate(row_values):
            for j, col_value in enumerate(col_values):
                case_best = []
                for case_name in cases:
                    candidates = [row for row in all_rows_v5 if row["case"] == case_name and row[row_key] == row_value and row[col_key] == col_value]
                    if candidates:
                        case_best.append(max(candidates, key=lambda row: row["P_valid"])["P_valid"])
                if case_best:
                    heat[i, j] = float(np.mean(case_best))
        return heat

    def v5_save_heatmap(heat, row_labels, col_labels, title, filename):
        fig, ax = plt.subplots(figsize=(10, 5))
        im = ax.imshow(heat, cmap="viridis", vmin=0, vmax=1, aspect="auto")
        ax.set_xticks(np.arange(len(col_labels)))
        ax.set_xticklabels(col_labels, rotation=25, ha="right")
        ax.set_yticks(np.arange(len(row_labels)))
        ax.set_yticklabels(row_labels)
        for i in range(heat.shape[0]):
            for j in range(heat.shape[1]):
                if np.isfinite(heat[i, j]):
                    ax.text(j, i, f"{heat[i, j]:.3f}", ha="center", va="center", color="white" if heat[i, j] < 0.65 else "black", fontsize=9)
        ax.set_title(title)
        fig.colorbar(im, ax=ax, label="mean best P_valid")
        fig.tight_layout()
        fig.savefig(PARAM_OUTPUT_DIR / f"{filename}.pdf", bbox_inches="tight")
        fig.savefig(PARAM_OUTPUT_DIR / f"{filename}.png", dpi=200, bbox_inches="tight")
        plt.close(fig)

    heat_mode_mixer = v5_mean_best_for_pair("transition_mode", "mixer_method", METHOD_MODES, MIXER_METHODS_V5)
    v5_save_heatmap(heat_mode_mixer, METHOD_MODES, MIXER_METHODS_V5, "V5 mean best P_valid by transition mode and mixer", "v5_all_config_transition_mixer_heatmap")
    heat_mode_method = v5_mean_best_for_pair("transition_mode", "method", METHOD_MODES, methods)
    v5_save_heatmap(heat_mode_method, METHOD_MODES, methods, "V5 mean best P_valid by transition mode and optimization method", "v5_all_config_transition_method_heatmap")
    heat_mixer_method = v5_mean_best_for_pair("mixer_method", "method", MIXER_METHODS_V5, methods)
    v5_save_heatmap(heat_mixer_method, MIXER_METHODS_V5, methods, "V5 mean best P_valid by mixer and optimization method", "v5_all_config_mixer_method_heatmap")

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    axes[0].bar(x, [row["theta_over_pi"] for row in best_rows_v5], color="#3498db")
    axes[1].bar(x, [row["beta_over_pi"] for row in best_rows_v5], color="#9b59b6")
    axes[2].bar(x, [row["r"] for row in best_rows_v5], color="#f39c12")
    for ax, title, ylabel in zip(axes, ["theta/pi", "beta/pi", "repetitions r"], ["theta/pi", "beta/pi", "r"]):
        ax.set_xticks(x)
        ax.set_xticklabels([c.replace("_", "\n", 2) for c in cases], rotation=45, ha="right")
        ax.set_title(title)
        ax.set_ylabel(ylabel)
    fig.suptitle("V5 best full-sweep parameters by use case", fontsize=15)
    fig.tight_layout(rect=[0, 0.02, 1, 0.94])
    fig.savefig(PARAM_OUTPUT_DIR / "v5_all_config_best_parameters_by_case.pdf", bbox_inches="tight")
    fig.savefig(PARAM_OUTPUT_DIR / "v5_all_config_best_parameters_by_case.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

    mixer_summary_rows = []
    for case_name in cases:
        for mixer_method in MIXER_METHODS_V5:
            candidates = [row for row in all_rows_v5 if row["case"] == case_name and row["mixer_method"] == mixer_method]
            if candidates:
                mixer_summary_rows.append(max(candidates, key=lambda row: row["P_valid"]))

    fig, ax = plt.subplots(figsize=(13, 5))
    width = 0.24
    for mi, mixer_method in enumerate(MIXER_METHODS_V5):
        mixer_rows = [next(row for row in mixer_summary_rows if row["case"] == case and row["mixer_method"] == mixer_method) for case in cases]
        offset = (mi - 1) * width
        ax.bar(x + offset, [row["P_valid"] for row in mixer_rows], width=width, label=mixer_method)
    ax.set_xticks(x)
    ax.set_xticklabels([c.replace("_", "\n", 2) for c in cases], rotation=45, ha="right")
    ax.set_ylabel("best P_valid")
    ax.set_ylim(0, 1.10)
    ax.set_title("V5 mixer method comparison by use case")
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(PARAM_OUTPUT_DIR / "v5_mixer_methods_best_by_case.pdf", bbox_inches="tight")
    fig.savefig(PARAM_OUTPUT_DIR / "v5_mixer_methods_best_by_case.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

    print("\nV5 parameter-method comparison over all case studies and transition modes")
    print("case | mixer | mode | method | theta/pi | beta/pi | r | P_valid | delta_P_vs_exp")
    print("-----|-------|------|--------|----------|---------|---|---------|---------------")
    for row in all_rows_v5:
        print(f"{row['case']} | {row['mixer_method']} | {row['transition_mode']} | {row['method']} | {row['theta_over_pi']:.3f} | {row['beta_over_pi']:.3f} | {int(row['r'])} | {row['P_valid']:.4f} | {row['delta_P_vs_exp']:+.4f}")
    print(f"\nSaved CSV: {csv_path}")
    print(f"Saved figure: {PARAM_OUTPUT_DIR / 'v5_param_methods_all_cases.pdf'}")
    print(f"Saved mixer figure: {PARAM_OUTPUT_DIR / 'v5_mixer_methods_best_by_case.pdf'}")
else:
    print("No V5 Analysis 6 rows were generated.")


[V5] Case 01_1d_tiny_single_gap: N=[6], M=[2], W=5, valid=1, uniform=0.2000
      delta_cost: P_valid=0.808261, theta=2.617994, beta=1.256637, r=48
      descent: P_valid=0.563533, theta=0.523599, beta=0.628319, r=4
      valid_boundary: P_valid=0.768116, theta=1.675516, beta=1.518436, r=56
      oriented_valid: P_valid=0.863791, theta=0.523599, beta=0.680678, r=57
[V5] Case 02_1d_main_reference: N=[8], M=[2], W=7, valid=2, uniform=0.2857
      delta_cost: P_valid=0.788246, theta=0.733038, beta=1.361357, r=59
      descent: P_valid=0.852866, theta=0.314159, beta=0.471239, r=15
      valid_boundary: P_valid=0.656485, theta=0.000000, beta=0.261799, r=55
      oriented_valid: P_valid=0.878844, theta=0.209440, beta=0.157080, r=37
[V5] Case 03_1d_two_free_regions: N=[10], M=[3], W=8, valid=3, uniform=0.3750
      delta_cost: P_valid=0.816923, theta=1.884956, beta=1.466077, r=39
      descent: P_valid=0.840733, theta=2.303835, beta=1.361357, r=54
      valid_boundary: P_valid=0.839568, theta

/var/folders/tp/l0dbjbqx2bdg68w801r6g6pw0000gn/T/ipykernel_26871/2024971860.py:291: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  fig.tight_layout()


      valid_boundary: P_valid=0.473405, theta=1.780236, beta=0.994838, r=4
      oriented_valid: P_valid=0.770458, theta=0.209440, beta=0.209440, r=16
[V5] Case 06_2d_tiny_corner_block: N=[3, 3], M=[2, 2], W=4, valid=1, uniform=0.2500
      delta_cost: P_valid=0.432443, theta=2.932153, beta=0.418879, r=40
      descent: P_valid=0.863912, theta=0.942478, beta=0.471239, r=38
      valid_boundary: P_valid=0.311139, theta=1.884956, beta=0.785398, r=57
      oriented_valid: P_valid=0.843204, theta=0.314159, beta=0.209440, r=33
[V5] Case 07_2d_small_diagonal_block: N=[4, 4], M=[2, 2], W=9, valid=1, uniform=0.1111
      delta_cost: P_valid=0.326623, theta=3.036873, beta=0.157080, r=16
      descent: P_valid=0.510077, theta=1.256637, beta=1.204277, r=45
      valid_boundary: P_valid=0.348536, theta=3.036873, beta=1.361357, r=46
      oriented_valid: P_valid=0.671782, theta=0.837758, beta=0.366519, r=8
[V5] Case 08_2d_medium_clustered_obstacles: N=[5, 5], M=[2, 2], W=16, valid=9, uniform=0.5625